# Besoin client 1 : Visualisation sur carte

In [1]:
# Import des packages
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap, MarkerCluster

## Chargement des données


In [2]:
df = pd.read_csv('IRVE_clean_IA.csv', sep=',' , encoding='utf-8')

FileNotFoundError: [Errno 2] No such file or directory: '/IRVE_clean_IA.csv'

## Extraction des données d'intérêt



In [ ]:
# Extraction des données d'intérêt
columns = [
    'implantation_station',
    'puissance_nominale',
    'nbre_pdc',
    'consolidated_latitude',
    'consolidated_longitude'
]

# On récrée une base de données avec seulement les colonnes qui nous intéressent
df_visu = df[columns].copy()

# Pas de nettoyage nécessaire, il n'y a aucune valeur manquante dans ces colonnes

## Visualisation des données (graphiques)

In [ ]:
sns.set_theme(style="whitegrid")

# Création d'une figure avec 2 graphiques côte à côte
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Graphique sur les types d'implantation (variable qualitative)
# On utilise un diagramme en barres et on trie par ordre décroissant
ordre_implantation = df_visu['implantation_station'].value_counts().index
sns.countplot(data=df_visu, y='implantation_station', order=ordre_implantation, palette="viridis", ax=axes[0])

axes[0].set_title("Distribution des types d'implantation", fontweight='bold', fontsize=14)
axes[0].set_xlabel("Nombre de stations", fontsize=12)
axes[0].set_ylabel("Type d'implantation", fontsize=12)


# Graphique des tranches de puissance

# On crée des catégories (tranches) basées sur les standards du marché
tranches = [0, 7.4, 22, 50, 150, 400]
labels = ['Lente (<7kW)', 'Standard (7-22kW)', 'Accélérée (22-50kW)', 'Rapide (50-150kW)', 'THP (>150kW)']

# On ajoute une colonne temporaire dans notre tableau pour ces tranches
df_visu['tranche_puissance'] = pd.cut(df_visu['puissance_nominale'], bins=tranches, labels=labels, right=True)

# On trace un countplot (comme pour les implantations)
sns.countplot(data=df_visu, x='tranche_puissance', palette="magma", ax=axes[1])

axes[1].set_title("Distribution par tranches de puissance", fontweight='bold', fontsize=14)
axes[1].set_xlabel("Tranche de puissance ", fontsize=12)
axes[1].set_ylabel("Nombre de stations", fontsize=12)

# On incline légèrement le texte de l'axe X pour que les labels ne se chevauchent pas
axes[1].tick_params(axis='x', rotation=15)

# Ajustement automatique des marges pour un affichage propre
plt.tight_layout()
plt.show()

## Visualisation sur carte selon le type d'implantation / puissance nominale

In [ ]:


# Initialisation de la carte (centrée sur la France)
carte_bornes = folium.Map(location=[46.603354, 1.888334], zoom_start=6, tiles="CartoDB positron")

df_reduce = df_visu.sample(15000,random_state=42)

# Création du groupe de clusters pour regrouper les points proches
marker_cluster = MarkerCluster(name="Stations IRVE").add_to(carte_bornes)

# Dictionnaire de couleurs pour le type d'implantation
couleurs_implantation = {
    'Voirie': 'blue',
    'Parking public': 'green',
    'Parking privé réservé à la clientèle': 'orange',
    'Parking privé à usage public': 'red',
    'Station dédiée à la recharge rapide' : 'yellow'
}

# Boucle de placement des points
for idx, row in df_reduce.iterrows():
    # Définition de la couleur (Gris si l'implantation est inconnue/autre)
    couleur = couleurs_implantation.get(row['implantation_station'], 'gray')

    # Calcul du rayon dynamique selon la puissance
    # On fixe une taille minimum de 3 et un maximum de 15 pour éviter des cercles immenses
    puissance = row['puissance_nominale']
    rayon_dynamique = max(3, min(puissance / 5, 15))

    # Ajout du marqueur
    folium.CircleMarker(
        location=[row['consolidated_latitude'], row['consolidated_longitude']],
        radius=rayon_dynamique,
        color=couleur,
        weight=1,
        fill=True,
        fill_color=couleur,
        fill_opacity=0.7,
        tooltip=(
            f"<div style='font-family: Arial;'>"
            f"<b>{row['implantation_station']}</b><br>"
            f"Puissance : <b>{puissance} kW</b><br>"
            f"</div>"
        )
    ).add_to(marker_cluster)

# Affichage de la carte interactive
display(carte_bornes)

## Heatmap : densité des bornes

In [ ]:
# Initialisation de la carte
carte_chaleur = folium.Map(location=[46.603354, 1.888334], zoom_start=6, tiles="CartoDB positron")

# Préparation des données pour la HeatMap
heat_data = df_reduce[['consolidated_latitude', 'consolidated_longitude', 'nbre_pdc']].values.tolist()

# Création et paramétrage de la couche de chaleur
HeatMap(
    heat_data,
    radius=15,          # Taille du halo de chaleur autour de chaque point
    blur=20,            # Flou pour permettre la fusion visuelle des halos proches
    max_val=df_visu['nbre_pdc'].max(),
    min_opacity=0.3
).add_to(carte_chaleur)

# Affichage de la carte
display(carte_chaleur)